# Deploying NVIDIA Nemotron 3.5 Transcribe with vLLM

This notebook prepares an OpenAI-compatible vLLM client for the BF16, FP8, or NVFP4 checkpoint and demonstrates transcription, context biasing, translation, English-only evaluated audio understanding, and optional word timestamps.

The model is an **offline** speech-and-audio model. Its internal audio chunks do not make it a streaming recognizer.

## Prerequisites

- Linux or WSL2 and a supported NVIDIA GPU.
- BF16 validated starting point: one H100 80 GB.
- FP8/NVFP4 validated starting point: one RTX PRO 6000 Blackwell 96 GB; NVFP4 requires Blackwell.
- vLLM 0.32.0 and a compatible NeMo Speech revision.
- Access to the selected Hugging Face checkpoint.
- A pre-existing Jupyter kernel with Python 3.11+ and `openai==3.13.0` for the client-only code cells.
- A local WAV, FLAC, MP3, M4A, or OGG recording. Audio is decoded to 16 kHz mono by the model integration.

> This notebook pins a matched post-rename NeMo Speech and model revision set. Current checkpoint configs use `Nemotron3_5TranscribeForConditionalGeneration`; pre-rename and post-rename components cannot be mixed. The included pair has not been independently GPU-validated; test it before production use and check the selected model card for the complete hardware matrix and current compatibility notes.

## Create the serving environment

Run these commands in a terminal. Keep the environment separate from any NeMo training environment.

```bash
git clone https://github.com/NVIDIA-NeMo/Speech.git nemo-speech
cd nemo-speech
export NEMO_SPEECH_REV=cbc617bb94a31451fe0a3503741e2a8e7568ff22
git checkout --detach "$NEMO_SPEECH_REV"
uv venv --python 3.13 .venv
UV_PROJECT_ENVIRONMENT=$PWD/.venv uv sync --extra asr --extra vllm --locked --no-install-project
uv pip install --python .venv/bin/python --no-deps -e .
# PyAV decodes compressed audio formats, such as M4A, without a system FFmpeg.
uv pip install --python .venv/bin/python --no-deps av==19.0.2
uv pip install --python .venv/bin/python --no-deps opencv-python-headless==5.0.0.93
source .venv/bin/activate
```

These commands create the server environment. Its locked Speech `vllm` extra includes `openai==3.13.0`, but it does not install or register Jupyter or `ipykernel`. The notebook therefore expects the pre-existing client kernel listed above; provision and register that kernel under your normal notebook dependency policy rather than mutating the locked server environment implicitly. Quantized checkpoints compile kernels on first startup and require CUDA 13 `nvcc` on `PATH`. Treat first-start compilation as cold-start time, not serving latency.

## Choose a checkpoint source

By default, the generated server command uses the selected Hugging Face model ID and downloads the revision pinned once in the profile cell below. To pre-download, use that exact ID and revision with `hf download`, set both `MODEL_VARIANT` and an absolute `MODEL_DIR` in the notebook environment, and rerun the profile cell. The cell validates the local `config.json`, renamed Transcribe architecture, and quantization scheme before emitting a command. Keep authentication in the Hugging Face CLI or process environment; do not put a token in this notebook. Reconfirm the revision hashes against the public repositories before use.

## Select a model-card launch profile

Set `MODEL_VARIANT` to `bf16`, `fp8`, or `nvfp4`. The cell prints a server command; run that command in a terminal from the NeMo Speech environment.

In [ ]:
import json
import os
import shlex
from pathlib import Path

model_variant_override = os.environ.get("MODEL_VARIANT")
MODEL_VARIANT = (model_variant_override or "bf16").lower()
PROFILES = {
    "bf16": {
        "model_id": "nvidia/NVIDIA-Nemotron-3.5-Transcribe-30B-A4B",
        "revision": "525be0eeaa30b1b0782fe1c96d172e0c012a2ec6",
        "gpu_memory": 0.94,
        "max_num_seqs": 192,
        "audio_limit": '{"audio":{"count":6,"length":120000000}}',
        "kv_cache": "auto",
        "blackwell_kernels": False,
    },
    "fp8": {
        "model_id": "nvidia/NVIDIA-Nemotron-3.5-Transcribe-30B-A4B-FP8",
        "revision": "94edf6deb1b42bdae3529b010e283de855dc473e",
        "gpu_memory": 0.85,
        "max_num_seqs": 64,
        "audio_limit": '{"audio":{"count":1,"length":57600000}}',
        "kv_cache": "auto",
        "blackwell_kernels": True,
    },
    "nvfp4": {
        "model_id": "nvidia/NVIDIA-Nemotron-3.5-Transcribe-30B-A4B-NVFP4",
        "revision": "499c4e5aa456ef25a36def3033f206e8c07e62eb",
        "gpu_memory": 0.85,
        "max_num_seqs": 64,
        "audio_limit": '{"audio":{"count":1,"length":57600000}}',
        "kv_cache": "fp8",
        "blackwell_kernels": True,
    },
}
if MODEL_VARIANT not in PROFILES:
    raise ValueError(f"MODEL_VARIANT must be one of {tuple(PROFILES)}, got {MODEL_VARIANT!r}")
profile = PROFILES[MODEL_VARIANT]
MODEL_ID = profile["model_id"]
model_dir_override = os.environ.get("MODEL_DIR")
if model_dir_override:
    if model_variant_override is None:
        raise ValueError("Set MODEL_VARIANT explicitly when MODEL_DIR is set.")
    model_dir = Path(model_dir_override).expanduser().resolve()
    config_path = model_dir / "config.json"
    if not config_path.is_file():
        raise FileNotFoundError(f"MODEL_DIR does not contain config.json: {model_dir}")
    model_config = json.loads(config_path.read_text(encoding="utf-8"))
    if model_config.get("model_type") != "nemo_speechlm":
        raise ValueError(f"MODEL_DIR is not a NeMo SpeechLM checkpoint: {model_dir}")
    expected_architecture = ["Nemotron3_5TranscribeForConditionalGeneration"]
    actual_architecture = model_config.get("architectures")
    if actual_architecture != expected_architecture:
        raise ValueError(
            f"MODEL_DIR architectures={actual_architecture!r}; this cookbook pins the post-rename "
            f"NeMo Speech loader and requires {expected_architecture!r}. Download the matching current "
            "checkpoint instead of mixing pre-rename and post-rename components."
        )
    expected_quant_algo = {"bf16": None, "fp8": "FP8", "nvfp4": "MIXED_PRECISION"}[MODEL_VARIANT]
    quantization_config = model_config.get("quantization_config") or {}
    actual_quant_algo = quantization_config.get("quant_algo")
    if actual_quant_algo != expected_quant_algo:
        raise ValueError(
            f"MODEL_DIR quant_algo={actual_quant_algo!r}, but MODEL_VARIANT={MODEL_VARIANT!r} "
            f"requires {expected_quant_algo!r}."
        )
    MODEL_SOURCE = str(model_dir)
else:
    MODEL_SOURCE = MODEL_ID
MODEL_ID, MODEL_SOURCE, profile["revision"]


In [ ]:
environment = [
    "export VLLM_PLUGINS=nemo_speechlm",
    "export VLLM_MAX_AUDIO_DECODE_DURATION_S=3600",
    "export VLLM_MAX_AUDIO_CLIP_FILESIZE_MB=180",
    "export VLLM_USE_V2_MODEL_RUNNER=0",
]
if profile["blackwell_kernels"]:
    environment.extend([
        "export PATH=/usr/local/cuda/bin:$PATH",
        "export VLLM_DISABLED_KERNELS=FlashInferFP8ScaledMMLinearKernel",
    ])

revision_arg = "" if model_dir_override else f" --revision {profile['revision']}"
command = [
    f"vllm serve {shlex.quote(MODEL_SOURCE)}{revision_arg}",
    f"  --served-model-name {shlex.quote(MODEL_ID)}",
    "  --host 127.0.0.1",
    "  --trust-remote-code --dtype bfloat16",
    f"  --kv-cache-dtype {profile['kv_cache']} --gpu-memory-utilization {profile['gpu_memory']}",
    f"  --max-model-len 65536 --max-num-batched-tokens 4096 --max-num-seqs {profile['max_num_seqs']}",
    "  --enable-chunked-prefill --no-enable-prefix-caching",
    "  --use-replayssm --mamba-backend flashinfer --mamba-cache-mode none",
    "  --mm-processor-cache-gb 0 --no-async-scheduling",
    f"  --limit-mm-per-prompt '{profile['audio_limit']}'",
    "  --speculative-config '{\"method\":\"mtp\",\"num_speculative_tokens\":8}'",
    "  --reasoning-parser nemotron_v3",
    "  --trust-request-mm-kwargs",
    "  --middleware nemo.collections.speechlm2.vllm.salm.ctc_serving.ctc_timestamp_middleware",
]
print("\n".join(environment))
print(" \\\n".join(command))


Wait for the server to report readiness on `127.0.0.1:8000`. The command binds loopback explicitly because the API has no authentication in this example. It uses the vLLM 0.32 ReplaySSM path required for native MTP on this hybrid backbone. The V1 model runner and `--trust-request-mm-kwargs` allow per-request CTC timestamp capture. For FP8, verify ModelOpt FP8 detection, an FP8 E4M3 KV cache, and 128 FP8 audio-encoder linear layers. For NVFP4, verify mixed ModelOpt detection, the explicit FP8 KV cache, and the FP8 audio encoder.

The command uses native MTP8. DFlash2 is an alternative, not an additional mode, and its exact K and CTC-disable settings differ by checkpoint. Follow the selected model card rather than modifying the command by analogy.

## Connect to the server

The default stays on loopback. Setting `OPENAI_BASE_URL` to another host transmits the entire base64-encoded recording to that service; use only a trusted endpoint with TLS, authentication, and appropriate data-handling controls.

In [ ]:
from openai import OpenAI

BASE_URL = os.environ.get("OPENAI_BASE_URL", "http://127.0.0.1:8000/v1")
client = OpenAI(base_url=BASE_URL, api_key=os.environ.get("OPENAI_API_KEY", "EMPTY"))
available = [model.id for model in client.models.list().data]
assert MODEL_ID in available, f"{MODEL_ID} is not served; available={available}"
available


## Prepare audio and a request helper

Set `AUDIO_PATH` to audio you are allowed to process. The notebook does not download or redistribute a sample recording. Request bodies send the recording to `OPENAI_BASE_URL`.

In [ ]:
import base64
import mimetypes

AUDIO_PATH = Path(os.environ.get("AUDIO_PATH", "audio.wav")).expanduser()
if not AUDIO_PATH.is_file():
    raise FileNotFoundError("Set AUDIO_PATH to a local WAV, FLAC, MP3, M4A, or OGG recording.")

def audio_data_url(path: Path) -> str:
    media_type = mimetypes.guess_type(path.name)[0] or "audio/wav"
    payload = base64.b64encode(path.read_bytes()).decode("ascii")
    return f"data:{media_type};base64,{payload}"

def ask_audio(
    prompt: str,
    *,
    audio_path: Path = AUDIO_PATH,
    system_prompt: str = "",
    reasoning: bool = False,
    timestamps: bool = False,
    max_tokens: int = 20480,
):
    messages = []
    if system_prompt:
        messages.append({"role": "system", "content": system_prompt})
    messages.append({
        "role": "user",
        "content": [
            {"type": "text", "text": prompt},
            {"type": "audio_url", "audio_url": {"url": audio_data_url(audio_path)}},
        ],
    })
    extra_body = {
        "chat_template_kwargs": {"enable_thinking": reasoning},
        "skip_special_tokens": False,
        "spaces_between_special_tokens": False,
        "seed": 0,
    }
    if reasoning:
        extra_body.update(top_k=-1, thinking_token_budget=17408)
        extra_body["chat_template_kwargs"]["reasoning_budget"] = 16384
    if timestamps:
        extra_body["mm_processor_kwargs"] = {"capture_ctc_timestamps": True}
    return client.chat.completions.create(
        model=MODEL_ID,
        messages=messages,
        temperature=0.6 if reasoning else 0,
        top_p=0.95 if reasoning else 1.0,
        max_tokens=max_tokens,
        stream=False,
        n=1,
        extra_body=extra_body,
    )


The examples below display raw generated text. For transcription, translation, and language identification, the model cards document a generation-loop cleanup guardrail used for reported evaluations. Apply it only where the selected card recommends it; do not apply it blindly to reasoning output, where it can damage legitimate formatting such as table rules. The implementation is not duplicated here so the model card remains the canonical implementation and attribution source.

A dense one-hour conversation can need about 20,000 output tokens. For long recordings, raise `max_tokens` or remove it, so that the transcript is not truncated.

## Basic transcription

Use explicit wording and formatting instructions when the transcript style matters.

In [ ]:
response = ask_audio(
    "Audio transcription. Language: as spoken. Wording: verbatim, including fillers and repetitions. "
    "Formatting: punctuation and capitalization. Output: transcript only."
)
print(response.choices[0].message.content)


## Context-biased transcription

Supply only relevant names and terms. Context is a hint, not a guarantee that each term occurs.

In [ ]:
context_terms = ["Nemotron", "NeMo Speech", "DFlash2"]
prompt = "Transcribe with punctuation and capitalization. Context terms: " + ", ".join(context_terms) + "."
response = ask_audio(prompt)
print(response.choices[0].message.content)


## Speech translation

Check the model card's language matrix before depending on a language pair.

In [ ]:
target_language = "English"
response = ask_audio(f"Please translate the given speech to {target_language}.")
print(response.choices[0].message.content)


## Spoken-language identification

Provide the complete set of labels that your application accepts.

In [ ]:
language_labels = ["Arabic", "English", "French", "German", "Hindi", "Spanish"]
response = ask_audio(
    "Identify the spoken language. Respond with exactly one label from "
    + ", ".join(language_labels)
    + " and nothing else."
)
print(response.choices[0].message.content)


## Audio understanding with reasoning

Reasoning is for questions about the recording, not ordinary transcription. It uses a larger token budget and non-greedy sampling. Audio understanding, reasoning, and spoken-instruction following were evaluated only in English and may not work reliably in other languages.

In [ ]:
response = ask_audio("What is happening in this audio?", reasoning=True)
print(response.choices[0].message.content)


## Word timestamps and speaker attribution

This requires the CTC middleware in the server command. It is unavailable when serving through the current DFlash2 path. Always inspect `error` before consuming alignments.

In [ ]:
speaker_prompt = (
    "Transcribe this audio with punctuation and capitalization. Include all spoken words and verbal "
    "fillers. Write numbers in their standard written form. Put speaker tags <spk:0> <spk:1> <spk:2> "
    "<spk:3> on the transcription."
)
response = ask_audio(speaker_prompt, timestamps=True)
payload = response.model_dump()
alignment = payload.get("ctc_timestamps")
if alignment is None:
    raise RuntimeError("The response has no ctc_timestamps; check that the middleware is enabled.")
if alignment.get("error") is not None:
    raise RuntimeError(alignment["error"])
print(response.choices[0].message.content)
for word in alignment.get("words", []):
    print(word["word"], word["start"], word["end"], word.get("speaker"))


## Submit multiple recordings concurrently

Set `AUDIO_PATHS` to a comma-separated list. Client concurrency is distinct from the engine's `max-num-seqs`; validate memory and latency with the recording lengths used by your application.

In [ ]:
from concurrent.futures import ThreadPoolExecutor

audio_paths = [Path(item.strip()).expanduser() for item in os.environ.get("AUDIO_PATHS", "").split(",") if item.strip()]
if audio_paths:
    missing = [str(path) for path in audio_paths if not path.is_file()]
    if missing:
        raise FileNotFoundError(f"Missing audio files: {missing}")
    with ThreadPoolExecutor(max_workers=min(4, len(audio_paths))) as pool:
        responses = list(pool.map(
            lambda path: ask_audio("Transcribe the following audio.", audio_path=path),
            audio_paths,
        ))
    for path, item in zip(audio_paths, responses, strict=True):
        print(f"{path}: {item.choices[0].message.content}")
else:
    print("Set AUDIO_PATHS to run the concurrent-request example.")


## Before production use

1. Pin the NeMo Speech and vLLM revisions and record the GPU, driver, CUDA, model revision, and server flags.
2. Validate cold start separately from warmed latency; quantized kernels compile on first use.
3. Evaluate the exact prompts, audio lengths, languages, concurrency, and timestamp mode used by the application.
4. Where the selected model card recommends it, apply the generation-loop guardrail to direct speech-task output; do not blindly apply it to reasoning output. Check for truncation, alignment errors, OOMs, and KV-cache preemptions.
5. Review the selected model card's license, safety, privacy, bias, and task-specific limitations.
6. Stop the vLLM terminal process when finished to release GPU memory.